## 一.为什么我们使用Transformer而不是RNN

我们知道transformer模型一开始只是为了解决机器翻译问题，那么在transformer出现以前，是使用的什么呢。根据之前的学习，我们知道使用的是RNN(循环神经网络)，我已经手写过一次RNN(在pytorch仓库中)，对RNN粗略的了解是：它设置了多个隐藏层用来对输入文本进行处理，每一层的输出将作为下一层的输入，解码器只通过最后一次整合了所有前文输入的结果作出对应的输出，这样可以避免逐词翻译，实现综合上下文翻译，但是这有一个致命的缺点————实际上，我们舍弃了中间的隐藏层，只使用最后的结果作为编码的输入，这会过于依赖当前隐藏层的状态，容易导致上下文缺失，在复杂文本中，依赖关系可能跨越句子的很长距离，这个时候RNN并不能很好的处理上下文

## 二.Transformer如何工作

简单来说，在即将生成输出文本时，编码器可以访问前面所有的输入词元，同时根据注意力权重衡量每一个词元对当前的重要性(这里涉及到Q，K，V，即查询向量，键向量，值向量)这三者作为词元的属性，QK得到重要性后，再将V值加权输入

## 三.自注意力机制

这是transformer的核心内容，要讲清楚，自注意力机制不同于传统的注意力机制，后者的注意力可能更多在输入和输出序列之间，但是自注意力机制通过关联单个输入序列中的不同位置来计算注意力权重，它可以评估并学习输入本身各个部分之间的关系和依赖，比如句子之间的单词，图片中的像素，下面我们从简单的无可训练权重的简单自注意力机制开始

### Part1：简单无训练权重的自注意力机制实现

In [3]:
import torch
from sklearn.externals.array_api_compat.dask.array.linalg import diagonal

inputs=torch.tensor([[0.43,0.15,0.89],[0.55,0.87,0.66],[0.57,0.85,0.64],[0.22,0.58,0.33],[0.77,0.25,0.10],[0.05,0.80,0.55]])#这里有六行三维的向量，分别假设其代表Your journey starts with one step六个单词
#实现自注意力机制的第一步是计算中间值w，即注意力分数，这个表示查询向量与各个单词的相关度，计算方法就是用查询向量与嵌入向量做点乘
#这里以第二个单词作为查询向量试试
query=inputs[1]#注意Q出现了
attn_scores_2=torch.empty(inputs.shape[0])#输入为6*3的向量，这里是为了创建长度为6的一维空列表，储存六个单词的注意力分数
for i,x_i in enumerate(inputs):
    attn_scores_2[i]=torch.matmul(query,x_i)
print(attn_scores_2)


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


不难看出，在六个单词中第二个单词的注意力分数最高，这是当然的因为我们就是取的它自己来查询，这就是利用了向量的点积，就是将各个位置的元素乘起来求和，是一种将矢量转化为标量的方法，点积越大，说明两个向量对齐的程度越高，接下来我们利用归一化，将所有分数变为和为1的数据,使用我们熟悉的softmax函数,非常好，pytorch自带了softmax

In [4]:
attn_scores_2=torch.softmax(attn_scores_2,dim=0)
print("Attention weights:",attn_scores_2)
print("Sum:",attn_scores_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


下面一步，是将所有前文的嵌入向量与其注意力分数相乘，最后相加，得到上下文向量

In [5]:
query=inputs[1]
context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_scores_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


接下来我们实现所有词元的query计算

In [6]:
attn_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j]=torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


上面的代码可以用矩阵乘法代替for循环

In [7]:
attn_scores=inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [8]:
attn_weight=torch.softmax(attn_scores,dim=1)
print(attn_weight)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


接下来就是将注意力权重与嵌入向量相乘

In [9]:
all_context_vecs=attn_weight @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Part2：带有可训练权重参数的自注意力机制实现

我们只是在无训练权重参数的基础上，加上了三个可训练的权重矩阵，查询(query),键(key),值(value),对于所有的词元，我们将键矩阵与嵌入向量矩阵相乘，得到键向量，在将嵌入向量与值矩阵相乘，得到值向量，而对于将要进行查询操作的词元，我们还需要将其嵌入向量与查询矩阵相乘得到查询向量

In [10]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2#输入的维度为三，输出维度为二，但是一般的gpt输入与输出维度是相同的，这里只是为了便于理解
torch.manual_seed(123)
W_query=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)
W_key=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)
W_value=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)
query_2=x_2 @ W_query#这里依旧是把第二个单词作为查询
key_2=x_2 @ W_key
value_2=x_2 @ W_value
print(query_2,key_2,value_2)
keys=inputs @ W_key#计算所有词元的值与键矩阵
values=inputs @ W_value
print("keys.shape:",keys.shape)
print("values.shape:",values.shape)

tensor([-1.1729, -0.0048]) tensor([-0.1142, -0.7676]) tensor([0.4107, 0.6274])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


在计算完所有词元的键矩阵之后，我们将所要查询的词元的查询向量与它们的键矩阵相乘(点积),就可以得到注意力分数

In [11]:
attn_scores_2=query_2 @keys.T
print(attn_scores_2)

tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809])


下面就是将注意力分数转换为注意力权重，不过此时，是将注意力分数除以键向量的嵌入维度的平方根来缩放,这是因为在GPT中一般嵌入向量的维度非常大，导致点积的结果也非常大，这个时候如果直接使用softmax函数，那么会得到很小(接近于零)的注意力权重，这会导致梯度过小，甚至消失，严重影响训练效果，因此这种自注意力机制也被称作缩放点积注意力机制

In [12]:
d_k=keys.shape[-1]
attn_weight_2=torch.softmax(attn_scores_2/d_k**0.5,dim=-1)
print(attn_weight_2)

tensor([0.1704, 0.1611, 0.1652, 0.1412, 0.2505, 0.1117])


接下来是将值向量加权求和得到上下文向量

In [13]:
context_vec_2=attn_weight_2 @ values
print(context_vec_2)

tensor([0.2854, 0.4081])


重新梳理一下：在这个阶段，首先我们有三个参数表，分别是query，key，value矩阵，我们要将输入词元的嵌入向量与每一个key，value向量做矩阵乘法得到每个token的key向量和value向量，然后在将查询token的嵌入向量与query相乘得到查询向量，将查询向量与前文每个token的键向量做点积得到注意力分数，进行归一化后得到每个token的注意力权重，再将所有token的值向量与权重相乘再相加起来，就得到最终的上下文向量

接下来将其封装成一个简单的python类

In [14]:
import torch.nn as nn
class SelfAttension_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query=nn.Parameter(torch.randn(d_in,d_out))
        self.W_key=nn.Parameter(torch.randn(d_in,d_out))
        self.W_value=nn.Parameter(torch.randn(d_in,d_out))
    def forward(self,x):
        keys=x @ self.W_key
        queries=x @ self.W_query
        values=x @ self.W_value
        attn_scores=queries @ keys.T
        attn_weight=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vec=attn_weight @ values
        return context_vec

torch.manual_seed(123)
sa_v1=SelfAttension_v1(d_in,d_out)
print(sa_v1(inputs))



tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]], grad_fn=<MmBackward0>)


下面用pytorch的线性层实现自注意力类

In [15]:
class SelfAttension_v2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weight = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec_2 = attn_weight @ values
        return context_vec_2
torch.manual_seed(123)
sa_v2=SelfAttension_v2(d_in,d_out)
print(sa_v2(inputs))



tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


明显看到这两种方法的结果并不一样，是因为在v1中我们采取的是正态分布初始化矩阵，而v2中采取的是一种更复杂的方式，如果想要更改至相同可以加上下列代码(注意，由于nn.linear是以转置的方式储存矩阵，所以更换参数时也要加上转置)

In [16]:
sa_v1.W_query.data = sa_v2.W_query.weight.data.T
sa_v1.W_key.data = sa_v2.W_key.weight.data.T
sa_v1.W_value.data = sa_v2.W_value.weight.data.T


### Part3：因果注意力

我们希望在给定词元时只根据序列先前和当下的输入来计算注意力分数，这个时候就需要用到因果注意力也叫做掩码注意力，将后文出现的词元分数调为0，并将剩下的注意力分数归一化

In [17]:
queries=sa_v2.W_query(inputs)
keys=sa_v2.W_key(inputs)
attn_scores=queries @ keys.T
attn_weight = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weight)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


接着我们使用Pytorch自带的tril函数实现第二步，创建一个对角线以上元素为零的掩码

In [18]:
context_length=attn_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


现在把这个掩码矩阵和注意力权重矩阵相乘

In [19]:
masked_simple=attn_weight*mask_simple
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


这样就是对角线以上全为零的注意力权重矩阵，但此时没有归一化

In [20]:
row_sums=masked_simple.sum(dim=-1,keepdim=True)#keepdim使得与原矩阵维度相同
mask_simple_norm=masked_simple/row_sums#利用广播机制使得每一个权重能够除以行总和
print(mask_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


以上就实现了归一化，但是还可以利用softmax更高效的实现掩码

In [21]:
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked=attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


不是将对角线右上方的数字设置为零，而是设置为负无穷，这样利用softmax可以完成归一化

In [22]:
attn_weight=torch.softmax(masked/keys.shape[-1]**0.5,dim=-1)
print(attn_weight)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


提到mask，不得不介绍一下dropout掩码，它在训练过程中随机抛弃掉部分隐藏层的信息，避免模型对某些隐藏层过于依赖形成过拟合，注意这个机制只在训练集中生效，测试时不用，而训练时一般是在计算注意力权重之后和在将这些权重用于向量值之后，我们接下来将在计算注意力权重之后应用dropout掩码

In [23]:
torch.manual_seed(123)
dropout=torch.nn.Dropout(0.5)#以0.5的比例舍弃数据
example=torch.ones(6,6)
print(dropout(example))


tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


可以看到大约有一半的元素被置为零，而剩下的元素则以1/0.5的比例扩大，这可以维持注意力权重的整体平衡，确保训练和推理过程中，注意力机制的平均影响保持一致

In [24]:
torch.manual_seed(123)
print(dropout(attn_weight))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


最后仍然是开发一个python类实现简单的因果注意力

In [25]:
batch=torch.stack((inputs,inputs),dim=0)
print(batch)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [35]:
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.d_out=d_out
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout=nn.Dropout(dropout)#添加了一个dropout层相较于自注意力机制
        self.register_buffer('mask',torch.ones(context_length,context_length))#自动将缓冲区和模型一起自动移动到合适的设备(GPU或者CPU)

    def forward(self,x):
        b,nums_tokens,d_in=x.shape
        keys=self.W_key(x)
        queries=self.W_query(x)
        values=self.W_value(x)
        attn_scores=queries @ keys.transpose(1,2)#将一二维度转置，将表示批的维度保持在第一个位置
        attn_scores.masked_fill(self.mask.bool()[:nums_tokens,:nums_tokens],-torch.inf)
        attn_weight=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weight=self.dropout(attn_weight)
        context_vec=attn_weight @ values
        return context_vec

In [38]:
torch.manual_seed(123)
context_length=batch.shape[1]
ca=CausalAttention(d_in,d_out,context_length,0.5)
context_vecs=ca(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[-0.8158, -0.1411],
         [-0.6920, -0.0972],
         [-0.4050, -0.1201],
         [-0.6902, -0.0969],
         [-0.5199, -0.0440],
         [-0.1417, -0.0505]],

        [[-0.7938, -0.2379],
         [-0.7858, -0.1145],
         [-0.3969,  0.0037],
         [-0.7704, -0.2374],
         [-0.7801, -0.1107],
         [-0.6749, -0.0984]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


### Part4:多头注意力

之前的因果注意力在运行时只有一组注意力权重按顺序处理输入，对于大模型来说，这不够快也不够准，首先通过堆叠多个因果注意力模块来构建多头注意力模块，但是还有一种更复杂，但是更高效的方法

简单来说，执行单头注意力层时，我们知道有三个参数矩阵，query，key，value。多头注意力的实质就是构建多个参数矩阵让它们并行运行，将每一个的结果相加，就得到最终的结果

In [39]:
class MultiHeadAttensionWrapper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([CausalAttention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)])#Modulelist是专门存放子模块的地方，而for循环是让x穿过Modulelist里面的所有层
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)#拼接函数，将所有并行的单头注意力结果拼接到一起，dim=-1就是从最后一个维度拼接，而最后一个维度是列


注意，结果向量的每一行表示的是一个词元的上下文向量，而列表示的是d_out所规定的嵌入维度，经过多头注意力处理后，我们的结果是在列方向上叠加，当num_heads=2，我们本身的嵌入向量维度又是2，那么最终的嵌入维度就是2*2=4，用代码展示一下

In [40]:
torch.manual_seed(123)
context_length=batch.shape[1]
d_in,d_out=3,2
mha=MultiHeadAttensionWrapper(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs=mha(batch)
print(context_vecs)
print("context_vecs.shape:",context_vecs.shape)

tensor([[[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


刚才的实现，实际上是创建了两个python类，causal和multi类，维护起来比较麻烦，事实上我们可以将它们合并成同一个类，将输入分为多个头，在计算注意力后合并这些头的结果

In [ ]:
class MultiHeadAttension(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0),"d_out must be divisible by num_heads"

        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads#减小投影维度以匹配所需的输出维度
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout=nn.Dropout(dropout)
        self.outproj=nn.Linear(dropout)#使用一个线性头来组合所有头的输出
        self.register_buffer( 'mask',torch.triu(torch.ones(context_length,context_length)),diagonal=1)
    def forward(self,x):
        b,nums_tokens,d_in=x.shape
        keys=self.W_key(x)
        queries=self.W_query(x)
        values=self.W_value(x)#此时的张量形状为(b,num_tokens,d_out)

        queries=queries.view(b,nums_tokens,self.num_heads,self.head_dim)
        values=values.view(b,nums_tokens,self.num_heads,self.head_dim)
        keys=keys.view(b,nums_tokens,self.num_heads,self.head_dim)
        #这里添加了一个num_heads维度，用来隐式地分隔矩阵，然后展开最后一个维度，view方法并没有真正的分割矩阵，而是说把最后的d_out视作num_heads和heads_dim两个维度

        keys=keys.transpose(1,2)
        queries=queries.transpose(1,2)
        values=values.transpose(1,2)#从(b,num_tokens,num_heads,head_dim)转换为(b,num_heads,num_tokens,head_dim)这样做的原因是pytorch矩阵乘法的特性，它只将最后的两个维度作为矩阵的行和列进行矩阵乘法，我们希望的是不同的头提取出不同的特征，而每一个头中的tokens之间互相做乘法，所以在这里，要讲tokens与heads的维度交换，同时由于GPU的特性，可以同时处理批次相同的数据，将heads提前可以实现GPU同时对多个头同时进行计算

        attn_scores=queries @ keys.transpose(2,3)#为了做矩阵乘法一定要对齐维度
        mask_bool=self.mask.bool()[:,:nums_tokens,:nums_tokens]

        attn_scores.masked_fill_(mask_bool,-torch.inf)#掩码填充注意力分数

        attn_weight=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weight=self.dropout(attn_weight)

        context_vec=(attn_weight @ values).transpose(1,2)#将刚才的token和head维度交换回顺序

        context_vec=context_vec.contiguous().view(b,nums_tokens,self.d_out)#将heads_dim和num_heads还原为d_out

        context_vec=self.out_proj(context_vec)#添加一个可选的线性投影
        return context_vec

区别于刚才的多头注意力，后者是将输入向量分别分给多张表分别处理(矩阵乘法)，最后加和，刚刚的这种实质上是首先创建一个相当大的权重矩阵，只将嵌入向量做一次矩阵乘法，得到注意力分数，这时候再将大矩阵分为多个小矩阵。这是为了榨取GPU的性能，让其只做一次矩阵乘法(矩阵乘法通常是非常耗时间的)，同时处理多个头的运行，